# Eval отчёт (ответы)
Быстрый просмотр результатов answer_eval: baseline (101), gold (50), Ollama sample20/gold50.
Файлы: eval/answer_eval_results*.jsonl, итог: eval/ANSWER_EVAL_SUMMARY.md, метрики ретривера: eval/rerank_results.md.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_palette('colorblind')

# Определяем базовый путь: если ноутбук лежит в eval/notebooks, берём выше
if '__file__' in globals():
    BASE = Path(__file__).resolve().parent.parent
else:
    cwd = Path.cwd()
    BASE = cwd.parent if cwd.name == 'notebooks' else cwd
EVAL_DIR = BASE if (BASE / 'answer_eval_results.jsonl').exists() else (BASE / 'eval')
FILES = {
    'deepseek': EVAL_DIR / 'answer_eval_results.jsonl',
    'deepseek_gold': EVAL_DIR / 'answer_eval_results_gold.jsonl',
    'ollama': EVAL_DIR / 'answer_eval_results_ollama_sample20.jsonl',
    'ollama_gold': EVAL_DIR / 'answer_eval_results_gold_ollama.jsonl',
}
for k, p in FILES.items():
    print(f"{k}: {p} -> {'есть' if p.exists() else 'MISSING'}")


In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(path, 'нет файла')
        return pd.DataFrame()
    rows = []
    for line in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            continue
    return pd.DataFrame(rows)

df_main = load_jsonl(FILES['deepseek'])
df_gold = load_jsonl(FILES['deepseek_gold'])
df_ollama = load_jsonl(FILES['ollama'])
df_ollama_gold = load_jsonl(FILES['ollama_gold'])

print('DeepSeek shape:', df_main.shape)
print('DeepSeek gold shape:', df_gold.shape)
print('Ollama shape:', df_ollama.shape)
print('Ollama gold shape:', df_ollama_gold.shape)


## Просмотр входных данных

In [ ]:
def show_head(df: pd.DataFrame, label: str):
    if df.empty:
        print(label + ': нет данных')
        return
    display(df.head()[['question','answer','answer_ref']])

show_head(df_main, 'DeepSeek')
show_head(df_gold, 'DeepSeek gold (50)')
show_head(df_ollama, 'Ollama sample20 (20)')
show_head(df_ollama_gold, 'Ollama gold (50)')


## Сводка метрик ответов
Используем avg_ref_sim и avg_refs_sim.

In [ ]:
def summarize(df: pd.DataFrame, label: str):
    if df.empty:
        print(label + ': нет данных')
        return
    cols = [c for c in ['ref_sim','refs_sim'] if c in df.columns]
    if not cols:
        print(label + ': нет столбцов ref_sim/refs_sim')
        return
    df_num = df[cols].apply(pd.to_numeric, errors='coerce')
    summary = df_num.agg(['mean','median','std']).T.reset_index().rename(columns={'index':'metric'})
    display(summary.style.format({'mean':'{:.4f}','median':'{:.4f}','std':'{:.4f}'}).set_caption(label))

summarize(df_main, 'DeepSeek (101)')
summarize(df_gold, 'DeepSeek gold (50)')
summarize(df_ollama, 'Ollama sample20 (20)')
summarize(df_ollama_gold, 'Ollama gold (50)')


## Распределения sim

In [ ]:
def plot_hists(df: pd.DataFrame, title: str):
    if df.empty:
        print(title + ': нет данных')
        return
    cols = [c for c in ['ref_sim','refs_sim'] if c in df.columns]
    if not cols:
        print(title + ': нет столбцов ref_sim/refs_sim')
        return
    df_num = df[cols].apply(pd.to_numeric, errors='coerce')
    df_num.plot(kind='hist', bins=20, alpha=0.7, title=title)
    plt.xlabel('similarity')
    plt.show()

plot_hists(df_main, 'DeepSeek: распределения sim')
plot_hists(df_gold, 'DeepSeek gold: распределения sim')
plot_hists(df_ollama, 'Ollama sample20: распределения sim')
plot_hists(df_ollama_gold, 'Ollama gold: распределения sim')


## Итоги ретривера (rerank_results.md)

In [ ]:
rr_path = EVAL_DIR / 'rerank_results.md'
if rr_path.exists():
    lines = rr_path.read_text(encoding='utf-8', errors='ignore').splitlines()
    # keep only table lines with pipes
    table_lines = [ln.strip() for ln in lines if ln.strip().startswith('|') and ln.strip().endswith('|')]
    if len(table_lines) >= 2:
        header_cells = [c.strip() for c in table_lines[0].split('|')[1:-1]]
        data = []
        for ln in table_lines[2:]:
            parts = [c.strip() for c in ln.split('|')[1:-1]]
            if len(parts) == len(header_cells):
                data.append(parts)
        if data:
            df_rr = pd.DataFrame(data, columns=header_cells)
            display(df_rr)
        else:
            print('No data rows parsed from rerank_results.md')
    else:
        print('Not enough table lines in rerank_results.md')
else:
    print(rr_path, 'file not found')
